# 🔄 Osnovni delovni tokovi agentov z Microsoft Foundry (Python)

## 📋 Vadnica o orkestraciji delovnih tokov

Ta zvezek predstavlja zmogljive zmogljivosti **Workflow Builder** v okviru Microsoft Agent Framework. Naučite se ustvarjati dovršene, večstopenjske delovne tokove agentov, ki lahko brezhibno obvladujejo kompleksne poslovne procese in usklajujejo več AI operacij.

> **Opomba o migraciji:** Ta primer je prej uporabljal GitHub Models. GitHub Models je zastarel (upokojen julija 2026), zato zdaj uporablja **Microsoft Foundry** preko `FoundryChatClient`, ki cilja na Azure OpenAI **Responses API**.

## 🎯 Cilji učenja

### 🏗️ **Arhitektura delovnega toka**
- **Workflow Builder**: Oblikovanje in orkestracija kompleksnih večstopenjskih procesov
- **Dogodkovno vodenje izvrševanja**: Upravljanje dogodkov delovnega toka in prehodov stanj
- **Vizualno oblikovanje delovnih tokov**: Ustvarjanje in vizualizacija struktur delovnega toka
- **Integracija Microsoft Foundry**: Izraba AI modelov v kontekstu delovnih tokov

### 🔄 **Orkestracija procesov**
- **Zaporedne operacije**: Povezovanje več nalog agentov v logičnem vrstnem redu
- **Pogojna logika**: Izvajanje odločitev in razvejitev delovnih tokov
- **Obravnava napak**: Zanesljiva obnova po napakah in odpornost delovnega toka
- **Upravljanje stanj**: Spremljanje in upravljanje stanja izvrševanja delovnega toka

### 📊 **Vzorce delovnih tokov v podjetjih**
- **Avtomatizacija poslovnih procesov**: Avtomatizacija kompleksnih organizacijskih delovnih tokov
- **Koordinacija več agentov**: Usklajevanje več specializiranih agentov
- **Razširljivo izvajanje**: Oblikovanje delovnih tokov za poslovne operacije na veliko
- **Nadzor & opazovanje**: Spremljanje uspešnosti in rezultatov delovnih tokov

## ⚙️ Zahteve in priprava okolja

### 📦 **Zahtevane odvisnosti**

Namestite Agent Framework z zmogljivostmi delovnih tokov:

```bash
pip install agent-framework -U
```

### 🔑 **Konfiguracija Microsoft Foundry**

Prijavite se z Azure CLI (`az login`), da se lahko `AzureCliCredential` overi, nato pa nastavite podrobnosti vašega projekta Microsoft Foundry.

**Nastavitev okolja (.env datoteka):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Uporabe v podjetjih**

**Primeri poslovnih procesov:**
- **Vključevanje strank**: Večstopenjsko preverjanje in nastavitve
- **Cevovod vsebin**: Avtomatizirano ustvarjanje, pregled in objava vsebin
- **Obdelava podatkov**: ETL delovni tokovi z AI-podprto transformacijo
- **Zagotavljanje kakovosti**: Avtomatizirani postopki testiranja in validacije

**Prednosti delovnih tokov:**
- 🎯 **Zanesljivost**: Deterministično izvrševanje z obnovo po napakah
- 📈 **Razširljivost**: Obvladovanje avtomatizacije velikih količin procesov
- 🔍 **Opazovanje**: Popolni revizijski sledovi in nadzor
- 🔧 **Vzdržljivost**: Vizualno oblikovanje in modularne komponente

## 🎨 Vzorci oblikovanja delovnih tokov

### Osnovna struktura delovnega toka
```mermaid
graph TD
    A[Začetek] --> B[Naloga agenta 1]
    B --> C{Odločitev}
    C -->|Uspeh| D[Naloga agenta 2]
    C -->|Neuspeh| E[Ravnatelj za napake]
    D --> F[Konec]
    E --> F
```

**Ključne sestavine:**
- **WorkflowBuilder**: Glavni orkestracijski motor
- **WorkflowEvent**: Obravnava dogodkov in komunikacija
- **WorkflowViz**: Vizualna predstavitev delovnega toka in razhroščevanje

Zgradimo vaš prvi inteligentni delovni tok! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
